<a href="https://colab.research.google.com/github/kutayeroglu/biomimetic-training/blob/main/eval_biomim.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Biomimetic Training Evaluation

This notebook evaluates model bias using texture-shape ablation analysis.

In [ ]:
# ============================================================================
# Configuration - Set all paths and parameters here
# ============================================================================
import os
import time
from pathlib import Path

# Google Drive paths
DRIVE_PATH = '/content/drive/MyDrive/colab_datasets/biomimetic_training'
RESULTS_SAVE_PATH = '/content/drive/MyDrive/colab_results/biomimetic_training/'

# Local paths
LOCAL_PATH = '/content/dataset'
REPO_PATH = '/content/biomimetic-training'

# File names
ZIP_FILE = "style-transfer-preprocessed-512-flat.zip"
CLASS_INDICES_FILE = "categories16_class_indices.pkl"

# Model configurations - Add or remove models here
# Each model will be evaluated and saved to a separate results file
MODELS = [
    {
        "checkpoint_file": "standard_checkpoint.pth",
        "model_identifier": "standard_alexnet"
    },
    {
        "checkpoint_file": "biomimetic_checkpoint.pth",
        "model_identifier": "biomimetic_alexnet"
    },
    {
        "checkpoint_file": "anti_biomimetic_checkpoint.pth",
        "model_identifier": "anti_biomimetic_alexnet"
    }
]

print("Configuration loaded.")
print(f"  Drive path: {DRIVE_PATH}")
print(f"  Local path: {LOCAL_PATH}")
print(f"  Number of models to evaluate: {len(MODELS)}")
for i, model in enumerate(MODELS, 1):
    print(f"    {i}. {model['checkpoint_file']} -> {model['model_identifier']}")
print(f"  Results will be saved to: {RESULTS_SAVE_PATH}")

In [ ]:
# ============================================================================
# Setup: Mount Google Drive
# ============================================================================
from google.colab import drive
drive.mount('/content/drive')

# Verify drive is accessible
if os.path.exists(DRIVE_PATH):
    print(f"✓ Drive mounted and accessible at {DRIVE_PATH}")
else:
    print(f"⚠ Warning: Drive path {DRIVE_PATH} not found. Please check the path.")

Mounted at /content/drive


In [ ]:
# ============================================================================
# Setup: Prepare Dataset
# ============================================================================
# Ensure the local directory exists
os.makedirs(LOCAL_PATH, exist_ok=True)

# Check if dataset already exists
if os.listdir(LOCAL_PATH):
    print(f"[SKIP] Dataset already exists at {LOCAL_PATH}. Skipping download/extraction.")
else:
    # Copy zip from Drive to local VM
    drive_zip_path = os.path.join(DRIVE_PATH, ZIP_FILE)
    
    if not os.path.exists(drive_zip_path):
        raise FileNotFoundError(
            f"Could not find {ZIP_FILE} at {DRIVE_PATH}\n"
            f"Please ensure the file exists in your Google Drive."
        )
    
    print(f"[INFO] Copying {ZIP_FILE} to local VM...")
    start_time = time.time()
    os.system(f'cp "{drive_zip_path}" /content/')
    duration = time.time() - start_time
    print(f"[SUCCESS] Copy complete. Time taken: {duration:.2f} seconds.")
    
    # Unzip to the local dataset folder
    print(f"[INFO] Unzipping dataset to {LOCAL_PATH}...")
    start_time = time.time()
    os.system(f'unzip -q /content/{ZIP_FILE} -d {LOCAL_PATH}')
    duration = time.time() - start_time
    print(f"[SUCCESS] Extraction complete. Time taken: {duration:.2f} seconds.")

# Verify dataset path
dataset_path = os.path.join(LOCAL_PATH, "style-transfer-preprocessed-512-flat")
if os.path.exists(dataset_path):
    num_images = len([f for f in os.listdir(dataset_path) if f.endswith('.png')])
    print(f"✓ Dataset ready: {num_images} PNG images found")
else:
    raise FileNotFoundError(f"Dataset directory not found: {dataset_path}")

In [ ]:
# ============================================================================
# Setup: Clone Repository and Install Dependencies
# ============================================================================
# Clone repository if it doesn't exist
if not os.path.exists(REPO_PATH):
    print(f"[INFO] Cloning repository...")
    os.system(f'git clone https://github.com/kutayeroglu/biomimetic-training.git {REPO_PATH}')
    print(f"✓ Repository cloned")
else:
    print(f"[SKIP] Repository already exists at {REPO_PATH}")

# Change to repo directory and add to path
os.chdir(REPO_PATH)
import sys
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)
print(f"✓ Working directory: {os.getcwd()}")

# Install dependencies
print(f"[INFO] Installing dependencies...")
os.system('pip install -q -r requirements.txt')
print(f"✓ Dependencies installed")

[INFO] Creating local directory: /content/dataset
[INFO] Copying style-transfer-preprocessed-512-flat.zip to local VM... (this may take a moment)
[SUCCESS] Copy complete. Time taken: 3.63 seconds.
[INFO] Unzipping dataset to /content/dataset...
[SUCCESS] Extraction complete. Time taken: 1.01 seconds.


In [ ]:
# ============================================================================
# Verify Required Files Exist
# ============================================================================
import pickle
import torch

# Check class indices file (shared across all models)
class_indices_path = os.path.join(DRIVE_PATH, CLASS_INDICES_FILE)
if not os.path.exists(class_indices_path):
    raise FileNotFoundError(f"Class indices file not found: {class_indices_path}")

# Quick verification of class indices
with open(class_indices_path, 'rb') as f:
    class_indices = pickle.load(f)
print(f"✓ Class indices loaded: {len(class_indices)} categories")
print(f"  Categories: {list(class_indices.keys())[:5]}...")  # Show first 5
del class_indices  # Free memory, will be loaded again in evaluation

# Verify all checkpoint files exist
print(f"\nVerifying checkpoint files for {len(MODELS)} models...")
missing_files = []
valid_models = []

for model in MODELS:
    checkpoint_file = model["checkpoint_file"]
    model_path = os.path.join(DRIVE_PATH, checkpoint_file)
    
    if not os.path.exists(model_path):
        print(f"  ✗ Missing: {checkpoint_file}")
        missing_files.append(checkpoint_file)
    else:
        # Verify checkpoint format (quick check without loading full model)
        try:
            checkpoint = torch.load(model_path, map_location='cpu')
            if isinstance(checkpoint, dict):
                if 'model_state_dict' not in checkpoint:
                    print(f"  ✗ Invalid format: {checkpoint_file} (missing 'model_state_dict' key)")
                    missing_files.append(checkpoint_file)
                else:
                    print(f"  ✓ Valid: {checkpoint_file} (State Dictionary format)")
                    valid_models.append(model)
                    del checkpoint  # Free memory
            else:
                print(f"  ⚠ Warning: {checkpoint_file} is not a dictionary format")
                valid_models.append(model)  # Still allow it, but warn
        except Exception as e:
            print(f"  ✗ Error loading {checkpoint_file}: {e}")
            missing_files.append(checkpoint_file)

# Update MODELS to only include valid models
if missing_files:
    print(f"\n⚠ Warning: {len(missing_files)} checkpoint file(s) not found or invalid:")
    for f in missing_files:
        print(f"    - {f}")
    print(f"\n  Continuing with {len(valid_models)} valid model(s)...")
    MODELS = valid_models

if len(MODELS) == 0:
    raise FileNotFoundError("No valid checkpoint files found. Cannot proceed with evaluation.")

print(f"\n✓ All required files verified. Ready to evaluate {len(MODELS)} model(s).")

Files loaded successfully.
Detected format: State Dictionary / Full Checkpoint
Keys found in checkpoint: dict_keys(['epoch', 'model_state_dict', 'optimizer_state_dict', 'val_acc'])


In [ ]:
# ============================================================================
# Run Evaluation for All Models
# ============================================================================
from src.eval.evaluate_bias import run_evaluation

# Prepare shared paths
data_path = os.path.join(LOCAL_PATH, "style-transfer-preprocessed-512-flat")
class_indices_path = os.path.join(DRIVE_PATH, CLASS_INDICES_FILE)

# Store results for each model
all_results = {}
result_files = []

print("=" * 80)
print(f"Starting Evaluation for {len(MODELS)} Model(s)")
print("=" * 80)
print(f"Data path: {data_path}")
print(f"Number of models: {len(MODELS)}")
print("=" * 80)

# Loop through each model configuration
for idx, model in enumerate(MODELS, 1):
    checkpoint_file = model["checkpoint_file"]
    model_identifier = model["model_identifier"]
    
    # Generate unique result filename for this model
    result_file = f"{model_identifier}_results.csv"
    result_path = result_file
    result_files.append(result_file)
    
    model_path = os.path.join(DRIVE_PATH, checkpoint_file)
    
    print(f"\n{'=' * 80}")
    print(f"Model {idx}/{len(MODELS)}: {checkpoint_file}")
    print(f"{'=' * 80}")
    print(f"  Checkpoint: {checkpoint_file}")
    print(f"  Model identifier: {model_identifier}")
    print(f"  Results will be saved to: {result_path}")
    print(f"{'=' * 80}\n")
    
    try:
        # Run evaluation for this model
        results = run_evaluation(
            model_path=model_path,
            data_path=data_path,
            class_indices_path=class_indices_path,
            model_file=model_identifier,
            result_path=result_path,
            overwrite=False,  # Set to True if you want to overwrite existing results
        )
        
        all_results[model_identifier] = results
        print(f"\n✓ Evaluation complete for {checkpoint_file}")
        
    except Exception as e:
        print(f"\n✗ Error evaluating {checkpoint_file}: {e}")
        print(f"  Continuing with remaining models...")
        # Remove from result_files if it wasn't created
        if result_file in result_files:
            result_files.remove(result_file)

print("\n" + "=" * 80)
print(f"All Evaluations Complete!")
print(f"  Successfully evaluated: {len(all_results)}/{len(MODELS)} model(s)")
print(f"  Result files created: {len(result_files)}")
for rf in result_files:
    print(f"    - {rf}")
print("=" * 80)

Cloning into 'biomimetic-training'...
remote: Enumerating objects: 190, done.
remote: Counting objects: 100% (190/190), done.
remote: Compressing objects: 100% (137/137), done.
remote: Total 190 (delta 95), reused 135 (delta 42), pack-reused 0 (from 0)
Receiving objects: 100% (190/190), 2.42 MiB | 33.94 MiB/s, done.
Resolving deltas: 100% (95/95), done.


In [ ]:
# ============================================================================
# Verify Results for All Models
# ============================================================================
import pandas as pd

# Get list of expected result files from the evaluation step
# If result_files wasn't defined (e.g., if Cell 6 wasn't run), generate from MODELS
if 'result_files' not in globals():
    result_files = [f"{model['model_identifier']}_results.csv" for model in MODELS]

print("=" * 80)
print("Verifying Results Files")
print("=" * 80)
print(f"Expected result files: {len(result_files)}")
print("=" * 80)

verified_files = []
missing_files = []

for result_file in result_files:
    print(f"\nChecking: {result_file}")
    print("-" * 80)
    
    if os.path.exists(result_file):
        try:
            results_df = pd.read_csv(result_file, index_col=0)
            
            print(f"✓ File exists and is readable")
            print(f"  Shape: {results_df.shape} (rows: categories, cols: ablation conditions)")
            print(f"  Number of categories: {len(results_df)}")
            print(f"  Number of columns: {len(results_df.columns)}")
            print(f"  Categories: {list(results_df.index)}")
            
            # Check data integrity
            missing_vals = results_df.isnull().sum().sum()
            empty_cells = (results_df == '').sum().sum()
            
            print(f"\n  Data integrity:")
            print(f"    Missing values: {missing_vals}")
            print(f"    Empty cells: {empty_cells}")
            
            # Show sample columns
            print(f"\n  First 5 columns:")
            for col in results_df.columns[:5]:
                print(f"    - {col}")
            
            file_size = os.path.getsize(result_file) / 1024
            print(f"\n  File size: {file_size:.1f} KB")
            
            verified_files.append(result_file)
            
        except Exception as e:
            print(f"✗ Error reading file: {e}")
            missing_files.append(result_file)
    else:
        print(f"✗ File not found")
        missing_files.append(result_file)

# Summary
print("\n" + "=" * 80)
print("Verification Summary")
print("=" * 80)
print(f"✓ Verified files: {len(verified_files)}/{len(result_files)}")
if verified_files:
    print(f"  Files:")
    for vf in verified_files:
        print(f"    - {vf}")

if missing_files:
    print(f"\n✗ Missing or invalid files: {len(missing_files)}")
    for mf in missing_files:
        print(f"    - {mf}")
print("=" * 80)

/content/biomimetic-training


In [ ]:
# ============================================================================
# Save All Results to Google Drive
# ============================================================================
# Create destination directory if it doesn't exist
os.makedirs(RESULTS_SAVE_PATH, exist_ok=True)

# Get list of result files to save
# If result_files wasn't defined (e.g., if Cell 6 wasn't run), generate from MODELS
if 'result_files' not in globals():
    result_files = [f"{model['model_identifier']}_results.csv" for model in MODELS]

print("=" * 80)
print("Saving Results to Google Drive")
print("=" * 80)
print(f"Destination: {RESULTS_SAVE_PATH}")
print(f"Number of files to save: {len(result_files)}")
print("=" * 80)

saved_files = []
failed_files = []

for result_file in result_files:
    source_path = result_file
    dest_path = os.path.join(RESULTS_SAVE_PATH, result_file)
    
    print(f"\nProcessing: {result_file}")
    print("-" * 80)
    
    if os.path.exists(source_path):
        try:
            print(f"[INFO] Copying {result_file} to Google Drive...")
            os.system(f'cp "{source_path}" "{dest_path}"')
            
            # Verify the file exists in Drive
            if os.path.exists(dest_path):
                file_size = os.path.getsize(dest_path) / 1024
                print(f"✓ Successfully saved to: {dest_path}")
                print(f"  File size: {file_size:.1f} KB")
                saved_files.append(result_file)
            else:
                print(f"✗ Error: File was not saved to Drive")
                failed_files.append(result_file)
        except Exception as e:
            print(f"✗ Error copying file: {e}")
            failed_files.append(result_file)
    else:
        print(f"✗ Error: Source file not found: {source_path}")
        failed_files.append(result_file)

# Summary
print("\n" + "=" * 80)
print("Save Summary")
print("=" * 80)
print(f"✓ Successfully saved: {len(saved_files)}/{len(result_files)} file(s)")
if saved_files:
    print(f"  Files:")
    for sf in saved_files:
        print(f"    - {sf}")

if failed_files:
    print(f"\n✗ Failed to save: {len(failed_files)} file(s)")
    for ff in failed_files:
        print(f"    - {ff}")
print("=" * 80)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 8.0 MB/s eta 0:00:00


In [ ]:
# ============================================================================
# Optional: Quick Preview of Results
# ============================================================================
# Uncomment to see a sample of the results for all models
# import pandas as pd
# 
# # Get list of result files
# if 'result_files' not in globals():
#     result_files = [f"{model['model_identifier']}_results.csv" for model in MODELS]
# 
# for result_file in result_files:
#     if os.path.exists(result_file):
#         results_df = pd.read_csv(result_file, index_col=0)
#         print(f"\n{result_file}:")
#         print(f"  Shape: {results_df.shape}")
#         print(f"  Sample (first category, first 5 columns):")
#         print(results_df.iloc[0, :5])